# Imports

In [1]:
!pip -q install openai pymupdf numpy gradio
# openai: embeddings and chat model calls
# pymupdf: reads PDFs and extracts text
# numpy: stores vectors and does fast similarity math
# gradio: web UI for chatting with RAG system

import os, re, glob, pickle
# os: file paths and folders
# re: text cleanup and tokenization (regex)
# glob: find all PDFs in a folder
# pickle: save/load Python objects (chunk metadata)

import numpy as np # used to store embeddings in arrays and do vector math quickly
import fitz  # fitz is the PyMuPDF API, it opens PDFs and extracts page text
import gradio as gr
from openai import OpenAI
from google.colab import drive, userdata

drive.mount("/content/drive")

PDF_DIR = "/content/drive/MyDrive/Tulane/Capstone/pdfs_for_chatbot" # path to pdfs folder
INDEX_DIR = "/content/drive/MyDrive/Tulane/Capstone/rag_index" # folder that stores the built index (embeddings and chunk metadata)
os.makedirs(INDEX_DIR, exist_ok=True) # creates INDEX_DIR if it does not exist already

EMBED_MODEL = "text-embedding-3-small" # embedding model name, turns text into vectors for retrieval
CHAT_MODEL  = "gpt-4o-mini" # chat model name, used to generate the final answer from retrieved context

emb_path  = os.path.join(INDEX_DIR, "X.npy") # file path to store/load the embeddings matrix (X)
meta_path = os.path.join(INDEX_DIR, "chunks.pkl") # file path to store/load chunk metadata (list of dicts)

# Initializes the OpenAI client using API key stored in Colab Secrets
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 45.0 MB/s eta 0:00:00
Mounted at /content/drive


# Extract page text from each PDF

In [2]:
# Extract text from each page of a PDF and return page records with metadata
def extract_pages(pdf_path: str):
    doc = fitz.open(pdf_path)  # open PDF
    pages = []

    for i in range(len(doc)):  # loop through pages
        text = doc.load_page(i).get_text("text") or ""
        text = re.sub(r"\s+", " ", text).strip()  # clean whitespace
        if text:  # skip empty pages
            pages.append({
                "doc_id": os.path.basename(pdf_path),
                "title": os.path.basename(pdf_path),
                "page": i + 1,  # 1-based page number
                "text": text,
            })

    return pages

# Find PDFs and build one combined list of all text pages across the library
pdf_paths = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
corpus_pages = []
for p in pdf_paths:
    corpus_pages.extend(extract_pages(p))

print("PDFs:", len(pdf_paths))
print("Pages with text:", len(corpus_pages))

PDFs: 55
Pages with text: 815


# Split each page into overlapping word chunks

In [3]:
# Tokenize text into simple "word" units and lowercase everything for consistency
# This keeps chunk boundaries stable and reduces noise from capitalization
def words(text: str):
    return re.findall(r"[A-Za-z0-9']+", text.lower())

# Split a long text into overlapping chunks of words
# chunk_size controls how much context each chunk holds
# overlap keeps some shared words between chunks so important info is not cut off at boundaries
def chunk_text(text: str, chunk_size=220, overlap=50):
    w = words(text)
    chunks = []
    start = 0
    while start < len(w):
        end = min(start + chunk_size, len(w))
        chunks.append(" ".join(w[start:end]))
        if end == len(w):
            break
        start = end - overlap
    return chunks

# Convert page records into chunk records, carrying over metadata for citations
# Each chunk gets a unique chunk_id with doc name + page + chunk index
def make_chunks(pages, chunk_size=220, overlap=50):
    chunks = []
    for pg in pages:
        for i, ch in enumerate(chunk_text(pg["text"], chunk_size, overlap)):
            chunks.append({
                "chunk_id": f"{pg['doc_id']}::p{pg['page']}::c{i}",
                "title": pg["title"],
                "page": pg["page"],
                "text": ch,
            })
    return chunks

chunks = make_chunks(corpus_pages, chunk_size=220, overlap=50)
print("Total chunks:", len(chunks))

Total chunks: 2693


# Embed each chunk into a normalized vector

In [4]:
def normalize(v):
    # L2-normalize vectors so dot product works as cosine similarity
    v = np.array(v, dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def embed_texts(texts):
    # Embed a batch of texts, normalize each embedding, return as a 2D matrix
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = [normalize(d.embedding) for d in resp.data]
    return np.vstack(vecs)

def build_or_load_index(chunks, batch_size=128):
    # Load cached embeddings + metadata if they match current chunk count, otherwise rebuild
    if os.path.exists(emb_path) and os.path.exists(meta_path):
        X = np.load(emb_path)
        with open(meta_path, "rb") as f:
            saved_chunks = pickle.load(f)
        if len(saved_chunks) == len(chunks):
            return X, saved_chunks
        print("Chunk count changed, rebuilding embeddings...")

    # Build embeddings in batches to avoid huge single requests
    X_list = []
    for i in range(0, len(chunks), batch_size):
        batch = [c["text"] for c in chunks[i:i+batch_size]]
        X_list.append(embed_texts(batch))
        if i % (batch_size * 10) == 0:
            print(f"Embedded {i}/{len(chunks)}")

    X = np.vstack(X_list)

    # Cache index to Drive so you do not re-embed every run
    np.save(emb_path, X)
    with open(meta_path, "wb") as f:
        pickle.dump(chunks, f)

    return X, chunks

# Build or load the embedding matrix (X) aligned with the chunks list
X, chunks = build_or_load_index(chunks)
print("X shape:", X.shape)

X shape: (2693, 1536)


# Retrieve top_k chunks

In [5]:
# Retrieval step: embed the query, score all chunks by cosine similarity, return top_k matches
def search(query, chunks, X, top_k=8):
    q = embed_texts([query])[0] # query embedding (normalized)
    sims = X @ q # cosine similarity vs every chunk (X rows are normalized)
    idx = np.argsort(-sims)[:top_k] # indices of highest scoring chunks
    return [(float(sims[i]), chunks[int(i)]) for i in idx]

# Context step: concatenate retrieved chunks into one prompt context within a word budget
def pack_context(results, max_words=900):
    used = 0
    parts = []
    for score, c in results:
        block = f"[source={c['chunk_id']}] {c['text']}"  # source tag enables citations
        n = len(words(block))
        if used + n > max_words:
            break
        parts.append(block)
        used += n
    return "\n\n".join(parts), used # returns context string + words used

# Generation

In [6]:
def answer_question(question, top_k=8, budget_words=900):
    # Full RAG pipeline for a single question:
    # 1. retrieve top_k relevant chunks
    # 2. pack them into a single context string within a word budget
    results = search(question, chunks, X, top_k=top_k)
    ctx, used_words = pack_context(results, max_words=budget_words)

    # System instructions force the model to stay grounded in the retrieved context
    system = (
        "You are a policy research assistant.\n"
        "Use ONLY the provided context.\n"
        "If the context does not contain the answer, say you do not know.\n"
        "Cite sources after relevant sentences using [source=...]."
    )

    # User message contains the packed context plus the question
    user = f"Context:\n{ctx}\n\nQuestion:\n{question}\n\nAnswer with citations."

    # Generation step: the chat model writes the final answer using the context
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0.2
    )

    # Extract model output text
    answer = resp.choices[0].message.content

    retrieved_preview = "\n".join(
        [f"{s:.3f} | {c['chunk_id']} | p{c['page']}" for s, c in results]
    )

    # Return the answer plus retrieval info for transparency
    return answer, retrieved_preview, used_words

# Gradio UI

In [7]:
# Fixed retrieval settings to keep the UI simple and consistent
FIXED_TOP_K = 8
FIXED_BUDGET_WORDS = 900

def chat_fn(message, history):
    # Gradio chat handler, runs RAG for the user message and formats the output
    ans, retrieved, used_words = answer_question(
        message,
        top_k=FIXED_TOP_K,
        budget_words=FIXED_BUDGET_WORDS
    )

    # Combine the model answer with a retrieval summary
    out = (
        f"{ans}\n\n"
        f"Retrieved (top {FIXED_TOP_K}), context words used: {used_words}\n"
        f"{retrieved}"
    )
    return out

demo = gr.ChatInterface(
    # Creates a chat-style web UI that calls chat_fn on each message
    fn=chat_fn,
    title="Education Policy RAG Chatbot",
    description="Ask questions about your PDF library. The system answers using retrieved PDF chunks with citations.",
    examples=[
        # Pre filled questions for quick testing
        "What are some of the most promising ways to improve the outcomes of low-income students?",
        "Under what conditions is early childhood education most effective?",
        "Which school resources are most important to academic achievement?",
    ],
)

demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0f370eb476cf84fb63.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0f370eb476cf84fb63.gradio.live
